In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [3]:
# Load source datasets
orders_reviews = spark.read.csv("orders_and_reviews_final_20260331_232353.csv", header=True, inferSchema=True)
customers_geo = spark.read.csv("customers_and_geolocation_final_20260403_135844.csv", header=True, inferSchema=True)
sellers_geo = spark.read.csv("sellers_and_geolocation_final_20260403_140220.csv", header=True, inferSchema=True)

# Join by customer_id and seller_id
orders_customers = orders_reviews.join(customers_geo, on="customer_id", how="inner")
orders_customers_sellers = orders_customers.join(sellers_geo, on="seller_id", how="inner")

orders_customers_sellers.printSchema()
orders_customers_sellers.show(5)

root
 |-- seller_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: date (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- volume_cm3: double (nullable = true)
 |-- review_score: double (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)


In [4]:
from pyspark.sql import functions as F
import csv
import os
import pandas as pd
from datetime import datetime

# Normalize delivery dates and compute on-time flag (1 = on time, 0 = late)
joined_df = orders_customers_sellers
joined_df = joined_df.withColumn(
    "order_delivered_customer_date",
    F.to_timestamp("order_delivered_customer_date")
 ).withColumn(
    "order_estimated_delivery_date",
    F.to_timestamp("order_estimated_delivery_date")
 ).withColumn(
    "delivered_on_time",
    F.when(
        F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"),
        F.lit(1)
    ).otherwise(F.lit(0))
 )

joined_df = joined_df.drop("seller_id").drop("customer_id") 
joined_pd = joined_df.toPandas()
output_dir = os.getcwd()

output_path = os.path.join(
    output_dir, f"table_final_{datetime.now():%Y%m%d_%H%M%S}.csv"
 )
joined_pd.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\table_final_20260403_143434.csv
